# The Transformer Architecture

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/transformers/03-transformer-architecture

A from-scratch, runnable implementation of the concepts in the lesson.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A full Transformer block in numpy

Assemble the pieces: multi-head self-attention + a position-wise feed-forward net, each wrapped in a **residual** connection and **LayerNorm** (pre-LN style).

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x)
    return e/e.sum(axis=axis, keepdims=True)

def layernorm(x, eps=1e-5):
    mu = x.mean(-1, keepdims=True); var = x.var(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

def mha(X, Wq, Wk, Wv, Wo, h, mask=None):
    T, d = X.shape; dk = d//h
    Q=(X@Wq).reshape(T,h,dk).transpose(1,0,2); K=(X@Wk).reshape(T,h,dk).transpose(1,0,2)
    V=(X@Wv).reshape(T,h,dk).transpose(1,0,2)
    s=Q@K.transpose(0,2,1)/np.sqrt(dk)
    if mask is not None: s=np.where(mask, s, -1e9)
    ctx=(softmax(s)@V).transpose(1,0,2).reshape(T,d)
    return ctx@Wo

def relu(x): return np.maximum(0, x)

class TransformerBlock:
    def __init__(self, d, h, d_ff, seed=0):
        rng=np.random.RandomState(seed); self.h=h
        self.Wq,self.Wk,self.Wv,self.Wo = (rng.randn(d,d)*0.1 for _ in range(4))
        self.W1=rng.randn(d,d_ff)*0.1; self.b1=np.zeros(d_ff)
        self.W2=rng.randn(d_ff,d)*0.1; self.b2=np.zeros(d)
    def __call__(self, X, mask=None):
        X = X + mha(layernorm(X), self.Wq,self.Wk,self.Wv,self.Wo, self.h, mask)  # residual
        ff = relu(layernorm(X)@self.W1 + self.b1)@self.W2 + self.b2
        return X + ff                                                            # residual

d_model, n_heads, d_ff, T = 32, 4, 128, 7
X = np.random.RandomState(1).randn(T, d_model)
block = TransformerBlock(d_model, n_heads, d_ff)
print('block output:', block(X).shape, '(same shape in, same shape out)')

## LayerNorm, worked by hand

LayerNorm standardizes across the **feature** dimension of each token (not the batch), then applies a learned scale $\gamma$ and shift $\beta$:

$$\mu = \tfrac{1}{d}\sum_i x_i,\quad \sigma^2 = \tfrac{1}{d}\sum_i (x_i-\mu)^2,\quad y = \gamma\,\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta.$$

We reproduce the lesson's tiny example $x=[2,4,4,4,5,5,7,9]$ in **pure stdlib** so the output is exact and deterministic. Expect $\mu=5$, $\sigma^2=4$, $\sigma=2$, normalized $=[-1.5,-0.5,-0.5,-0.5,0,0,1,2]$ (mean 0, var 1), and with $\gamma=2,\beta=1$: $[-2,0,0,0,1,1,3,5]$.

In [ ]:
import math

# --- LayerNorm worked by hand (pure stdlib, deterministic) ---
x = [2, 4, 4, 4, 5, 5, 7, 9]               # one token's d=8 feature vector
d = len(x)
mu = sum(x) / d                            # mean across features
var = sum((xi - mu) ** 2 for xi in x) / d  # population variance (as in LayerNorm)
std = math.sqrt(var)
norm = [(xi - mu) / std for xi in x]       # eps negligible here
gamma, beta = 2.0, 1.0                     # learned affine params (broadcast)
y = [gamma * n + beta for n in norm]

print("mu   =", mu)
print("var  =", var, " std =", std)
print("norm =", [round(v, 4) for v in norm])
print("mean(norm) =", round(sum(norm) / d, 12),
      " var(norm) =", round(sum(v ** 2 for v in norm) / d, 12))
print("y = gamma*norm + beta =", [round(v, 4) for v in y])

assert mu == 5.0 and var == 4.0 and std == 2.0
assert [round(v, 4) for v in norm] == [-1.5, -0.5, -0.5, -0.5, 0.0, 0.0, 1.0, 2.0]
assert [round(v, 4) for v in y] == [-2.0, 0.0, 0.0, 0.0, 1.0, 1.0, 3.0, 5.0]
print("LayerNorm hand-check: PASS")

## Counting an encoder layer's parameters

One encoder layer = attention + FFN + two LayerNorms (biases ignored):

- **Attention** $W_Q, W_K, W_V, W_O$, each $d_{\text{model}}\times d_{\text{model}}$ → $4\,d_{\text{model}}^2$
- **FFN** $W_1\ (d_{\text{model}}\times d_{ff})$ + $W_2\ (d_{ff}\times d_{\text{model}})$ → $2\,d_{\text{model}}\,d_{ff}$
- **2 LayerNorms** ($\gamma,\beta$ each length $d_{\text{model}}$) → $4\,d_{\text{model}}$

For $d_{\text{model}}=512,\ d_{ff}=2048,\ h=8$ this is $1{,}048{,}576 + 2{,}097{,}152 + 2{,}048 = 3{,}147{,}776 \approx 3.15$M. The head count $h$ does not change the total (per-head projections concatenate back to $d\times d$). FFN is $\approx 66.6\%$ of the parameters.

In [ ]:
# --- Encoder-layer parameter count (pure stdlib, deterministic) ---
# Local names (d_model_enc, etc.) so this demo doesn't clobber the d_model/d_ff/h
# used by the TransformerBlock stack defined above and stacked further below.
d_model_enc, d_ff_enc, h_enc = 512, 2048, 8

attn = 4 * d_model_enc ** 2       # Wq, Wk, Wv, Wo each d x d
ffn  = 2 * d_model_enc * d_ff_enc # W1 (d x d_ff) + W2 (d_ff x d)
ln   = 2 * (2 * d_model_enc)      # two LayerNorms, gamma+beta each length d
total = attn + ffn + ln

print(f"attention 4*d^2  = {attn:,}")
print(f"FFN 2*d*d_ff     = {ffn:,}")
print(f"2 LayerNorms 4*d = {ln:,}")
print(f"total per layer  = {total:,}  (~{total/1e6:.2f}M)")
print(f"FFN share        = {ffn/total*100:.1f}%   attention share = {attn/total*100:.1f}%")
print(f"6-layer encoder  ~ {6*total/1e6:.1f}M params (before embeddings)")

assert attn == 1_048_576 and ffn == 2_097_152 and ln == 2_048
assert total == 3_147_776
print("param-count hand-check: PASS")

## Stacking blocks = a deep Transformer

Real models stack dozens of identical blocks. Residuals keep activations stable as depth grows — we track the activation norm through the stack.

In [ ]:
blocks = [TransformerBlock(d_model, n_heads, d_ff, seed=i) for i in range(12)]
norms = [np.linalg.norm(X)]
h = X
for blk in blocks:
    h = blk(h); norms.append(np.linalg.norm(h))
plt.plot(norms, 'o-', color='#6366f1'); plt.xlabel('block depth'); plt.ylabel('||activations||')
plt.title('Residual connections keep a 12-block stack stable'); plt.show()

## Decoder-only forward pass (GPT-style)

Add token embeddings + positional encoding, run causal blocks, project to vocab logits, softmax → next-token distribution. Here, untrained, just to show the data flow and shapes.

In [ ]:
vocab, d_model, T = 50, 32, 7
rng = np.random.RandomState(2)
tok_emb = rng.randn(vocab, d_model)*0.1
def pos_enc(T, d):
    p=np.arange(T)[:,None]; i=np.arange(d)[None,:]; a=p/np.power(10000,(2*(i//2))/d)
    pe=np.zeros((T,d)); pe[:,0::2]=np.sin(a[:,0::2]); pe[:,1::2]=np.cos(a[:,1::2]); return pe

tokens = rng.randint(0, vocab, size=T)
Xh = tok_emb[tokens] + pos_enc(T, d_model)
mask = np.tril(np.ones((T, T))).astype(bool)
for blk in blocks:
    Xh = blk(Xh, mask=mask)
W_out = rng.randn(d_model, vocab)*0.1
logits = layernorm(Xh) @ W_out
probs = softmax(logits)
print('next-token distribution shape:', probs.shape)
print('predicted next token after position', T-1, '->', int(probs[-1].argmax()))

## Key takeaways

- A block alternates **attention** (mix tokens) and a **feed-forward** net (per-token compute).
- **Residuals + LayerNorm** are what make deep stacks trainable — activation norm stays bounded.
- A causal mask turns the encoder block into a GPT-style **decoder**.
- The full forward pass is: embed + position → N blocks → project → softmax over the vocab.

## ✏️ Your turn

### Exercise 1 — LayerNorm from scratch

LayerNorm standardizes across the **feature** dimension of each token:

$$\\hat{x} = \\frac{x - \\mu}{\\sqrt{\\sigma^2 + \\epsilon}}, \\quad \\mu = \\frac{1}{d}\\sum_i x_i, \\quad \\sigma^2 = \\frac{1}{d}\\sum_i (x_i - \\mu)^2$$

After normalization the output has mean $\\approx 0$ and variance $\\approx 1$ per row.
Implement it and verify on the hand-worked example from the lesson: $x = [2,4,4,4,5,5,7,9]$.

In [ ]:
import numpy as np

def layernorm_manual(x, eps=1e-5):
    """Normalize x (1-D array) to mean 0, var 1 using the LayerNorm formula.
    Returns the normalized array (no learnable gamma/beta here)."""
    # TODO(you): compute mu, sigma^2, return (x - mu) / sqrt(sigma^2 + eps)
    ...

In [ ]:
x = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])
out = layernorm_manual(x)

assert abs(out.mean()) < 1e-6, \
    "LayerNorm output must have mean ≈ 0"
assert abs(out.var() - 1.0) < 1e-4, \
    "LayerNorm output must have variance ≈ 1"

# Exact values from the lesson's worked example (mu=5, sigma=2)
expected = np.array([-1.5, -0.5, -0.5, -0.5, 0.0, 0.0, 1.0, 2.0])
assert np.allclose(out, expected, atol=1e-4), \
    "x=[2,4,4,4,5,5,7,9] should normalize to [-1.5,-0.5,-0.5,-0.5,0,0,1,2]"

# Edge case: single-feature vector (d=1) -- variance is exactly 0, epsilon prevents a divide-by-zero
out_single = layernorm_manual(np.array([5.0]))
assert out_single.shape == (1,)
assert np.allclose(out_single, 0.0, atol=1e-6), \
    "a single-feature vector has zero variance; normalized value collapses to 0"

# Edge case: constant vector (zero variance across several features)
out_const = layernorm_manual(np.array([3.0, 3.0, 3.0, 3.0]))
assert np.allclose(out_const, 0.0, atol=1e-2), \
    "a constant vector has zero variance; every normalized entry should be ~0"

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def layernorm_manual(x, eps=1e-5):
    mu  = x.mean()
    var = x.var()
    return (x - mu) / np.sqrt(var + eps)
```

</details>

### Exercise 2 — Residual connections

Each Transformer sublayer wraps its computation in a residual connection:

$$y = x + \\text{Sublayer}(x)$$

This seemingly simple addition is what makes deep stacks trainable: gradients flow
directly through the skip path without going through the sublayer. Implement the wrapper
and verify the two key properties: shape preservation, and identity when the sublayer outputs zeros.

In [ ]:
def residual_block(X, sublayer_fn):
    """Apply sublayer_fn to X and add the residual: return X + sublayer_fn(X)."""
    # TODO(you): one line
    ...

In [ ]:
rng3 = np.random.RandomState(42)
X3 = rng3.randn(6, 32)

out3 = residual_block(X3, lambda x: np.zeros_like(x))
assert out3.shape == X3.shape, "residual must preserve shape"
assert np.allclose(out3, X3), \
    "when the sublayer outputs zeros, residual_block(X, zeros) must return X exactly"

# With a real sublayer the output is X + sublayer(X), not just X
sublayer_output = rng3.randn(6, 32) * 0.1
out4 = residual_block(X3, lambda _: sublayer_output)
assert np.allclose(out4, X3 + sublayer_output), \
    "output should be X + sublayer(X)"

# Edge case: 1-D input (a single feature vector, not a (T, d) batch)
x1d = np.array([1.0, -2.0, 3.0])
assert np.allclose(residual_block(x1d, lambda x: x * 0), x1d), \
    "residual_block must work for 1-D inputs too"

# Edge case: single-token sequence (T=1)
X_t1 = np.array([[2.0, 4.0, -1.0]])
sub_out = np.array([[0.5, -0.5, 1.0]])
assert np.allclose(residual_block(X_t1, lambda _: sub_out), X_t1 + sub_out), \
    "residual_block must add correctly even for a single-token sequence"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def residual_block(X, sublayer_fn):
    return X + sublayer_fn(X)
```

</details>

### Exercise 3 — DML practice: `layer_normalization` for batched sequence data (DML #109)

DML #109's `layer_normalization(X, gamma, beta, epsilon=1e-5)` generalizes Exercise 1's 1-D
`layernorm_manual` to a full **batch of sequences**: `X` has shape `(batch, seq_len, d_model)`,
and normalization happens across the last axis (`d_model`) independently for every
`(batch, seq_len)` position, then scales by `gamma` and shifts by `beta` (both broadcastable
over that last axis) — the exact operation used inside `TransformerBlock` above.

In [ ]:
def layer_normalization(X, gamma, beta, epsilon=1e-5):
    """DML #109: LayerNorm over the LAST axis of a (batch, seq_len, d_model) tensor,
    with learnable gamma/beta broadcasting over that axis."""
    # TODO(you): mean/var over axis=-1 (keepdims=True), normalize, scale by gamma, shift by beta
    ...

In [ ]:
import numpy as np

# DML #109 -- exact test vectors from tests.json
np.random.seed(42)
X = np.random.randn(2, 2, 3)
gamma = np.ones(3).reshape(1, 1, -1); beta = np.zeros(3).reshape(1, 1, -1)
out109 = layer_normalization(X, gamma, beta)
expected109 = [[[0.474, -1.391, 0.917], [1.414, -0.707, -0.707]],
               [[1.132, 0.168, -1.3], [1.414, -0.705, -0.71]]]
assert np.allclose(out109, expected109, atol=1e-3)

np.random.seed(42)
X2 = np.random.randn(2, 3, 4)
gamma2 = np.ones(4).reshape(1, 1, -1) * 0.5; beta2 = np.ones(4).reshape(1, 1, -1)
out109b = layer_normalization(X2, gamma2, beta2)
expected109b = [[[0.886, 0.35, 1.013, 1.751], [0.537, 0.537, 1.73, 1.196], [0.708, 1.866, 0.715, 0.712]],
                [[1.7, 0.475, 0.582, 1.243], [0.8, 1.828, 0.881, 0.49], [1.727, 0.904, 1.047, 0.322]]]
assert np.allclose(out109b, expected109b, atol=1e-3)

# Edge case: single-batch, single-token sequence (1, 1, d) -- matches Exercise 1's hand example
X_single = np.array([[[2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]]])
out_single = layer_normalization(X_single, np.ones(8), np.zeros(8))
assert np.allclose(out_single[0, 0], [-1.5, -0.5, -0.5, -0.5, 0.0, 0.0, 1.0, 2.0], atol=1e-3), \
    "must match the Exercise 1 hand-worked example when reshaped to (1, 1, d)"

# Edge case: single feature dimension (d=1) -- zero variance, epsilon keeps it finite
X_d1 = np.random.randn(2, 3, 1)
out_d1 = layer_normalization(X_d1, np.ones(1), np.zeros(1))
assert np.allclose(out_d1, 0.0, atol=1e-6), "a single feature has zero variance; normalized to 0"

print("✅ Exercise 3 passed (DML #109 layer normalization for sequence data)")

<details>
<summary>💡 Show solution</summary>

```python
def layer_normalization(X, gamma, beta, epsilon=1e-5):
    mean = X.mean(axis=-1, keepdims=True)
    var = X.var(axis=-1, keepdims=True)
    X_norm = (X - mean) / np.sqrt(var + epsilon)
    return gamma * X_norm + beta
```

</details>

### Beyond this lesson: LayerNorm-free transformers (DML #128)

[DyT (Dynamic Tanh)](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/128_dynamic-tanh-normalization-free-transformer-activa)
is a 2024 proposal that replaces every LayerNorm in a Transformer block with
$\text{DyT}(x) = \gamma \odot \tanh(\alpha x) + \beta$ -- no mean or variance is computed at all,
just a single learned scalar $\alpha$ inside a squashing nonlinearity. See
`neural-networks/05-batchnorm-and-dropout.ipynb` for the full derivation and a worked example;
here it's just worth knowing LayerNorm is not the only way to keep activations well-scaled.